# Datamine RECMODWF Process Example

This notebook demonstrates how to configure and run the **`recmodwf`** process wrapper in `dmstudio`.

## Process Description

## RECMODWF

Compare the tonnes and grades of groups of cells in a block model with the tonnes and grades of volumes defined by wireframes. An example of how it might be used is to compare the optimised pushbacks defined in a block model against designed pushbacks defined by wireframes.

### MODEL File

The input model file for reconciliation. It must contain a field identifying the cells to be reconciled with the wireframe volumes.

For example, if the model has been created by Studio NPVS or Studio NPVS+, this field may be the optimally planned pushback identifier **PSB_PIT** or **PIT_NO**. The wireframe files could represent the planned pushbacks. The results would provide an indication of how close the designed pushbacks are to the optimised pushbacks.

## ### MODOUT

Optional output model file containing cell divisions defined by the input wireframes. This file will contain the **IDFIELD2** field with values derived from the input wireframe. If the input file already contains the **IDFIELD2** field its values are overwritten.

* **Note** (The previous values are saved in a field called **OldID2** _):

If a single value of **IDFIELD1** and **IDFIELD2** is defined using the @**VALUE** parameter then the output model will contain a **RECCODE** field with values of either 3, 4 or 5. These values are:

CODE=3, SOURCE=IDFIELD1 and IDFIELD2:

The total amount of material in the model with the same specified value of **IDFIELD1** AND **IDFIELD2**

CODE=4, SOURCE=IDFIELD1 Only:

The total amount of material in the model with a specified value of **IDFIELD1** and NOT with the specified value of **IDFIELD2**

CODE=5, SOURCE=IDFIELD2 Only:

The total amount of material in the model with a specified value of **IDFIELD2** and NOT with the specified value of **IDFIELD1**

### Results

Output results file containing tonnes and grades for categories of each unique value of **IDFIELD2**. The categories, with value of **RECCODE** from 1 to 5, are as follows:

CODE=1, SOURCE=IDFIELD1:

The total amount of material in the model with a specified value of **IDFIELD1**.

CODE=2, SOURCE=IDFIELD2:

The total amount of material in the model with a specified value of **IDFIELD2**.

CODE=3, SOURCE=IDFIELD1 and IDFIELD2:

The total amount of material in the model with the same specified value of **IDFIELD1** AND **IDFIELD2**.

CODE=4, SOURCE=IDFIELD1 Only:

The total amount of material in the model with a specified value of **IDFIELD1** and NOT with the specified value of **IDFIELD2**.

CODE=5, SOURCE=IDFIELD2 Only:

The total amount of material in the model with a specified value of **IDFIELD2** and NOT with the specified value of **IDFIELD1**.

[![](../Images/RECMODWF_2.png)](<javascript:void\(0\);>)

### Input Files:

* **model** (Block model):
  Input model file for reconciliation. It must contain a field identifying the planned
  cells to be reconciled with the wireframe volumes. If the model has been created by
  Studio NPVS or Studio NPVS+, this field may be the optimally planned pushback identifier

## **PSB_PIT**.

  Required=Yes

* **wiretr** (Wireframe triangles):
  Input wireframe triangle file used to define the mined volume(s). This should contain a
  field identifying the design to be reconciled.
  Required=Yes

* **wirept** (Wireframe points):
  Input wireframe point file used to to define the mine volume(s).
  Required=Yes

### Output Files:

* **modout** (Block Model):
  Output model file containing cell divisions defined by the input wireframes. This file
  will contain the **MINED** field with values derived from the input wireframe. If the
  input file already contains a MINED field its values are overwritten. The previous
  values are saved in a field called OldMined_
  Required=No

* **results** (Results):
  Output results file containing the reserve comparisons. This contains up to 5 records
  for every separate reconciled volume: Total Planned, Total Mined, Planned and Mined,
  Planned Only and Mined Only. Volumes are defined by the **PLANNED** and **MINED** fields
  and can be further broken down by the **KEY1** field and **BENCH** parameter.
  Required=Yes

### Fields:

* **planned** (Any : MODEL):
  Field in **MODEL** file used to group the planned blocks. If comparing wireframe
  designs with pushback reserves in a Studio NPVS(+) model this field may be **PSB_PIT.**
  Default=Undefined
  Required=Yes

* **mined** (Any : WIRETR, MODEL):
  Field in the **WIRETR** file defining the volumes to be compared to the corresponding
  **PLANNED** block model cells.
  Default=Undefined
  Required=Yes

* **key1** (Any : MODEL):
  Optional key field in the **MODEL** file used to categorize results (e.g. a Rock type
  field).
  Default=Undefined
  Required=No

* **density** (Any : MODEL):
  Density field in the **MODEL** file used to calculate tonnages.
  Default=DENSITY
  Required=No

* **grades** (Undefined : Undefined):
  Grade field in the model file
  Default=Undefined
  Required=No

### Parameters:

* **value**:
  Value of **PLANNED** and **MINED** fields to compare. If undefined or zero then all
  values of **MINED** field will be compared.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **modltype**:
  Type of wireframe model to be filled; one of the following options, with default of (1)
  :-
  Range=
  Values=
  Default=Solid 3d, interior to be filled with cells
  Required=No

* **factor**:
  Scaling factor to adjust the units of the Volume and Tonnage in the output files.
  Volume and Tonnage are divided by this factor.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **setabsnt**:
  Set to 1 to allow **[TONGRAD](<tongrad.md>)** to internally reset absent grade and
  Density values. If this is used, absent grade values are set to their default values. If
  the default value is absent grade values are set to zero. If Density values are absent
  the default DENSITY parameter value is used."
  Range=0, 1
  Values=0, 1
  Default=0
  Required=No

* **bench**:
  Set to 1 to categorize the reserve comparisons by benches.
  Range=
  Values=
  Default=Do not categorize by benches
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('recmodwf')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute recmodwf
print("Running recmodwf...")
dm_cmd.recmodwf(
    model_i='t_mod1',  # required
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    results_o=['t_recmodwf_out'],  # required
    planned_f='optional',  # required
    mined_f='optional',  # required
    # modout_o='t_recmodwf_out',  # optional
    # key1_f='optional',  # optional
    # density_f='optional',  # optional
    # grades_f=['optional'],  # optional
    # value_p='optional',  # optional
    # modltype_p='Solid 3d, interior to be filled with cells',  # optional
    # factor_p=1,  # optional
    # setabsnt_p=0,  # optional
    # bench_p='Do not categorize by benches',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("recmodwf execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_recmodwf_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")